# Information-Based Learning

### 4.2.2 Shannon's Entropy Model
$$H(t) = - \sum_{i=1}^{l} (P(t = i) \times \log_s(P(t = i)))$$


In [1]:
import math

def entropy(pt, base):
    return -sum(p * math.log(p, base) for p in pt)

p1 = [1.0]
p2 = [0.5, 0.5]
p3 = [0.99, 0.01]

print(entropy(p1, 2))
print(entropy(p2, 2))
print(entropy(p3, 2))

-0.0
1.0
0.08079313589591118


### 4.2.3 Information Gain


## 4.3 Standard Approach: The ID3 Algorithm

ID3 algorithm implementation:

In [ ]:
import numpy as np
import pandas as pd


def calculate_entropy(target_col):
    """Calculates the overall entropy of a target label array."""
    elements, counts = np.unique(target_col, return_counts=True)
    entropy = np.sum(
        [
            - (counts[i] / np.sum(counts)) * np.log2(counts[i] / np.sum(counts))
            for i in range(len(elements))
        ]
    )
    return entropy


def calculate_info_gain(data, split_attribute_name, target_name):
    """Calculates the Information Gain of a specific feature split."""
    # Total entropy before split
    total_entropy = calculate_entropy(data[target_name])

    # Calculate weighted entropy after split
    vals, counts = np.unique(data[split_attribute_name], return_counts=True)
    weighted_entropy = np.sum(
        [
            (counts[i] / np.sum(counts))
            * calculate_entropy(
                data[data[split_attribute_name] == vals[i]][target_name]
            )
            for i in range(len(vals))
        ]
    )

    # Information Gain formula
    information_gain = total_entropy - weighted_entropy
    return information_gain


def id3(data, original_data, features, target_attribute_name="Target"):
    """Recursively builds the decision tree using the ID3 algorithm."""
    # Condition 1: If all target values are the same, return that value (Leaf Node)
    if len(np.unique(data[target_attribute_name])) <= 1:
        return np.unique(data[target_attribute_name])[0]

    # Condition 2: If dataset is empty, return the majority target value from original dataset
    elif len(data) == 0:
        return np.unique(original_data[target_attribute_name])[
            np.argmax(np.unique(original_data[target_attribute_name], return_counts=True)[1])
        ]

    # Condition 3: If no features are left, return the majority target value of the current node
    elif len(features) == 0:
        return np.unique(data[target_attribute_name])[
            np.argmax(np.unique(data[target_attribute_name], return_counts=True)[1])
        ]

    # Build a subtree
    else:
        # Determine feature with the highest information gain
        item_values = [
            calculate_info_gain(data, feature, target_attribute_name)
            for feature in features
        ]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Initialize the tree structure
        tree = {best_feature: {}}

        # Remove the best feature from the feature list
        features = [i for i in features if i != best_feature]

        # Grow branches for each unique value of the selected feature
        for value in np.unique(data[best_feature]):
            sub_data = data[data[best_feature] == value]
            subtree = id3(sub_data, original_data, features, target_attribute_name)
            tree[best_feature][value] = subtree

        return tree

Calling the ID3 algorithm

In [ ]:
# Create a sample categorical dataset
data_dict = {
    "Outlook": [
        "Sunny",
        "Sunny",
        "Overcast",
        "Rain",
        "Rain",
        "Rain",
        "Overcast",
        "Sunny",
        "Sunny",
        "Rain",
    ],
    "Temperature": [
        "Hot",
        "Hot",
        "Hot",
        "Mild",
        "Cool",
        "Cool",
        "Cool",
        "Mild",
        "Cool",
        "Mild",
    ],
    "Humidity": [
        "High",
        "High",
        "High",
        "High",
        "Normal",
        "Normal",
        "Normal",
        "High",
        "Normal",
        "Normal",
    ],
    "Play": ["No", "No", "Yes", "Yes", "Yes", "No", "Yes", "No", "Yes", "Yes"],
}

df = pd.DataFrame(data_dict)
features = list(df.columns[:-1])  # Extract all columns except the target

# Generate the decision tree
resulting_tree = id3(df, df, features, target_attribute_name="Play")

import pprint

pprint.pprint(resulting_tree)